# Brain tumor segmentation from T1 to T2

How well does a model trained on T1-weighted MRI segment the same subjects on T2-weighted MRI? We compare two 50-epoch models. **RAW** sees normalized MRI intensity. **SSC4** sees four maps describing how similar each local patch is to its neighbors.

**In these fits, RAW Dice fell from 0.743 on T1 to 0.070 on T2. SSC4 scored 0.643 on T1 and 0.619 on T2.** SSC4 therefore transferred better across contrast in this task, at the cost of lower source T1 performance.

The endpoint is **whole tumor including edema**. T1 and T2 share a single reference mask for each of the 97 held-out subjects. The numeric report uses small files from `notebooks/data`. Saved illustration panels appear when the local bundle is available.

### Released models

Both downloads contain the epoch-50 model used for the validation scores below. The SHA256 values identify the extracted checkpoint files.

| Model and release page | Native model ZIP | Checkpoint SHA256 |
|---|---|---|
| [RAW release](https://github.com/bapierro/ml-biomed-project/releases/tag/brain-raw-t1-50ep-v1) | [Download RAW](https://github.com/bapierro/ml-biomed-project/releases/download/brain-raw-t1-50ep-v1/brain-raw-t1-50ep-v1-model.zip) | `93b5e54291b468912e9a8cc0f0a7a02345d8c026d3bff15d6e0e736c48997a97` |
| [SSC4 release](https://github.com/bapierro/ml-biomed-project/releases/tag/brain-ssc4-t1-50ep-v1) | [Download SSC4](https://github.com/bapierro/ml-biomed-project/releases/download/brain-ssc4-t1-50ep-v1/brain-ssc4-t1-50ep-v1-model.zip) | `c99b94fe065a79b8ef1e55ddee94d11b71caab1b74eea01adedc7afff0d64238` |

Section 5 shows how to load either model locally.

## 1. Data and source training

The local Medical Segmentation Decathlon Task01 files contain four channels: FLAIR (0), **T1w (1)**, contrast-enhanced T1 (2) and **T2w (3)**. Both models train on channel 1. Labels 1, 2 and 3 form one binary foreground. All modalities from a subject stay in the same split.

From the project root, launch with `uv run --locked --extra cpu --group ssc --group notebooks jupyter lab notebooks/brain-t1-t2.ipynb`.

In [ ]:
%matplotlib inline
import hashlib
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

PROJECT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/ml_biomed_project").is_dir()
)
REPORT_DIR = PROJECT / "notebooks" / "data"
study = json.loads((REPORT_DIR / "brain-t1-t2-study.json").read_text())
BROWSER_DIR = PROJECT / "data" / "brain-t1-t2-demo"


def verified_bytes(path, expected_sha):
    payload = path.read_bytes()
    if hashlib.sha256(payload).hexdigest() != expected_sha:
        raise ValueError(f"File differs from its recorded SHA256: {path.name}")
    return payload


def show_figure(fig, name):
    # Optional figure export: set BRAIN_FIGURE_DIR to a fresh output directory.
    destination = os.environ.get("BRAIN_FIGURE_DIR")
    if destination:
        folder = Path(destination)
        folder.mkdir(parents=True, exist_ok=True)
        for extension in ("png", "pdf"):
            path = folder / f"{name}.{extension}"
            if path.exists():
                raise FileExistsError(path)
            fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()


print(study["status_note"])

In [ ]:
design = study["design"]
audit = study["data_audit"]
display(
    pd.DataFrame(
        [
            {"Role": "T1 training", "Subjects": design["training_subjects"]},
            {"Role": "Paired T1/T2 held-out validation", "Subjects": design["held_out_subjects"]},
            {
                "Role": "Unlabeled challenge test — unused",
                "Subjects": audit["unlabeled_test_count"],
            },
        ]
    ).set_index("Role")
)
print(f"Data audit: {audit['status']} · {audit['cases_completed']} labeled subjects")
print("All: 240 × 240 × 155 voxels, 1 mm isotropic; image/reference grids agree.")
print("Selected channels are finite; all whole-tumor masks are nonempty.")
print("Exact T1/T2 duplicates:", audit["exact_channel_duplicates"])
display(pd.DataFrame(study["data_summary"]["foreground_ml"]).T.rename_axis("Role").round(3))
print("Foreground volumes above are in mL; descriptive QC, not selection criteria.")

The split was fixed by a hash of subject ID before fitting. Only the 387 training subjects’ T1 scans determined the native 2D plan. The models share the split, preprocessing, optimizer, augmentation and seed. Both use the fixed epoch-50 checkpoint.

RAW supplies one intensity channel. SSC4 compares local 3 × 3 patches at radius one and passes only four derived channels to the network. A contrast change can preserve some patch relationships while changing others, including those at a tumor boundary.

Each fitted model retains 12,500 updates. RAW resumed after epoch 27. Its interrupted run also executed 46 updates that the saved checkpoint did not retain, for 12,546 physical executions. Native T1 cropping omitted 95 labeled voxels from one training subject. All 97 held-out T1 subjects retained their full reference foreground.

## 2. Paired full-volume Dice

Each subject contributes one score under each model and sequence. The primary comparison is **RAW T2 minus RAW T1**. The four means show whether a smaller modality gap coincides with useful segmentation on both sequences.

In [ ]:
results_path = REPORT_DIR / study["report_files"]["results"]
results = None
if results_path.is_file() and study.get("results_sha256") and study.get("reconciliation_sha256"):
    results = json.loads(verified_bytes(results_path, study["results_sha256"]))
results_ready = results is not None and results.get("status") == "COMPLETE"
if not results_ready:
    state = results.get("status", "PENDING") if results else "PENDING"
    print(f"Full-volume results: {state}. No Dice scores or comparison chart available yet.")
else:
    if results["n_subjects"] != 97 or results["n_cells"] != 388:
        raise ValueError("Results do not cover the frozen 97-subject comparison")
    means = results["means"]
    scores = pd.DataFrame(
        [
            {"Recipe": recipe, "T1 Dice": means[f"{recipe}_T1"], "T2 Dice": means[f"{recipe}_T2"]}
            for recipe in design["recipes"]
        ]
    ).set_index("Recipe")
    display(scores.round(4))
    ax = scores.plot.bar(rot=0, figsize=(7, 4), color=["#536d8c", "#ba6843"])
    ax.set(
        ylabel="Mean full-volume Dice",
        ylim=(0, 1),
        title="Paired development validation · 97 subjects",
    )
    for bars in ax.containers:
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=9)
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)
    plt.tight_layout()
    show_figure(ax.figure, "absolute-dice")

In [ ]:
if results_ready:
    contrasts = results["paired_comparisons"]
    table = []
    for item in design["contrasts"]:
        estimate = contrasts[item["key"]]
        table.append(
            {
                "Contrast": item["label"],
                "Mean Dice change": estimate["difference"],
                "Unadjusted 95% CI": f"[{estimate['ci95'][0]:+.4f}, {estimate['ci95'][1]:+.4f}]",
                "Paired subjects": design["held_out_subjects"],
            }
        )
    display(pd.DataFrame(table).set_index("Contrast").round(4))
else:
    print("All five paired contrasts are pending; no intervals inferred from missing results.")

**The primary RAW T2−T1 change was −0.674 Dice (95% CI −0.711 to −0.631).** The analogous change for SSC4 was −0.024 (−0.047 to −0.001). SSC4 scored 0.549 higher on T2 and 0.100 lower on T1 than RAW.

The five prespecified contrasts use the same 10,000 bootstrap samples of held-out subjects (seed 20261104), with unadjusted 95% intervals. These intervals describe subject variability for the two fitted models.

### Where does the score change?

Each dot is one held-out subject. Its horizontal position is the T1 score and its vertical position is the T2 score. A dot below the diagonal means that subject scored lower on T2. Both panels use the same axes.

RAW scored lower on T2 in **96 of 97** subjects. SSC4 scored lower in 65, higher in 31 and identically in one. On T2, 77 RAW segmentations and two SSC4 segmentations had Dice below 0.10; both models produced nonempty masks on all subjects.

In [ ]:
if not results_ready:
    print("Paired subject plot pending final reconciliation.")
else:
    case_path = REPORT_DIR / study["report_files"]["case_dice"]
    verified_bytes(case_path, study["case_dice_sha256"])
    cases = pd.read_csv(case_path)
    counts = cases[["tp", "fp", "fn", "reference_voxels", "predicted_voxels"]]
    if (
        len(cases) != 388
        or cases.case_id.nunique() != 97
        or cases.duplicated(["case_id", "condition"]).any()
        or set(cases.condition) != set(design["conditions"])
        or not np.isfinite(counts.to_numpy()).all()
        or not ((counts >= 0) & (counts % 1 == 0)).all().all()
        or not (cases.tp + cases.fn == cases.reference_voxels).all()
        or not (cases.tp + cases.fp == cases.predicted_voxels).all()
        or not (cases.reference_voxels > 0).all()
        or not (cases.groupby("case_id").reference_voxels.nunique() == 1).all()
        or not np.allclose(
            cases.dice,
            2 * cases.tp / (cases.reference_voxels + cases.predicted_voxels),
            atol=1e-12,
            rtol=0,
        )
    ):
        raise ValueError("Expected 388 unique, valid count and Dice records")
    paired = cases.pivot(index="case_id", columns="condition", values="dice")
    if (
        paired.shape != (97, 4)
        or paired.isna().any().any()
        or not all(np.isclose(paired[c].mean(), means[c], atol=1e-12, rtol=0) for c in paired)
    ):
        raise ValueError("Paired subjects or scores disagree with the reconciled means")

    fig, axes = plt.subplots(
        1, 2, sharex=True, sharey=True, figsize=(8.5, 4.2), layout="constrained"
    )
    for ax, recipe, color in zip(axes, ("RAW", "SSC4"), ("#536d8c", "#168e99"), strict=True):
        x, y = paired[f"{recipe}_T1"], paired[f"{recipe}_T2"]
        ax.plot([0, 1], [0, 1], color="0.55", ls="--", lw=1, zorder=0)
        ax.scatter(x, y, s=21, color=color, alpha=0.65, linewidths=0)
        ax.scatter(
            x.mean(),
            y.mean(),
            marker="D",
            s=48,
            color="white",
            edgecolor="black",
            linewidth=1,
            label="Mean",
            zorder=4,
        )
        ax.set(
            xlim=(-0.025, 1.025),
            ylim=(-0.025, 1.025),
            aspect="equal",
            xlabel="T1w Dice",
            title=recipe,
        )
        ax.text(
            0.04,
            0.96,
            f"Lower on T2: {(y < x).sum()}/{len(x)}",
            transform=ax.transAxes,
            va="top",
            fontsize=9,
        )
        ax.set_xticks([0, 0.25, 0.5, 0.75, 1])
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
    axes[0].set_ylabel("T2w Dice")
    axes[1].legend(loc="lower right", frameon=False, fontsize=9)
    fig.suptitle("T1w and T2w scores for the same 97 validation subjects")
    show_figure(fig, "paired-dice")

## 3. Learning on source T1

Training and validation losses come from the native 2D trainer. **Validation patch Dice** measures sampled T1 patches. Full-volume Dice above measures one complete segmentation per held-out subject. The dashed line marks RAW’s resume from the epoch-27 checkpoint.

In [ ]:
history_path = REPORT_DIR / study["report_files"]["training"]
if not history_path.is_file():
    print("Training history: PENDING. No learning curves available yet.")
else:
    verified_bytes(history_path, study["training_history_sha256"])
    history = pd.read_csv(history_path)
    if set(history.columns) != set(study["training_columns"]):
        raise ValueError("Unexpected training-history schema")
    if history.duplicated(["recipe", "completed_epoch"]).any():
        raise ValueError("Duplicate recipe/epoch records")
    if not set(history.recipe) <= set(design["recipes"]):
        raise ValueError("Unexpected training recipe")
    if not history.completed_epoch.between(1, 50).all():
        raise ValueError("History extends beyond the frozen 50-epoch horizon")
    if results_ready and len(history) != 100:
        raise ValueError("The final report requires 50 unique epochs from each source model")
    fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
    for recipe, color in [("RAW", "#536d8c"), ("SSC4", "#168e99")]:
        rows = history.loc[history.recipe == recipe].sort_values("completed_epoch")
        print(f"{recipe}: {len(rows)}/50 completed epochs retained")
        for ax, column, label in zip(
            axes,
            ["train_loss", "val_loss", "patch_Dice"],
            ["Native training loss", "Native T1 validation loss", "Native T1 patch Dice"],
        ):
            if not rows.empty:
                ax.plot(rows.completed_epoch, rows[column], color=color, label=recipe)
            ax.set(ylabel=label, xlim=(1, 50))
            ax.grid(axis="y", alpha=0.2)
    if "recovery" in study:
        resumed_at = study["recovery"]["completed_epochs_restored"]
        for i, ax in enumerate(axes):
            ax.axvline(
                resumed_at + 0.5,
                color="#767676",
                lw=1,
                linestyle="--",
                label="RAW resumed after epoch 27" if i == 0 else "_nolegend_",
            )
    axes[0].set_title("Source T1 learning · terminal50 protocol", loc="left")
    axes[0].legend(frameon=False)
    axes[-1].set(xlabel="Completed epoch (1-based)", ylim=(0, 1))
    plt.tight_layout()
    show_figure(fig, "source-learning-curves")

## 4. Paired illustrations

Three held-out subjects were fixed before T2 inference using the 25th, 50th and 75th percentiles of original tumor volume. For each subject, both sequences use the same full-field axial slice with the most reference foreground. Ties use the first slice in RAS+ orientation. Each panel’s Dice is for the entire volume. These examples illustrate a range of tumor sizes.

In [ ]:
fixed_examples = ("BRATS_253", "BRATS_358", "BRATS_461")
selection_sha = "8672da369c76eefa75b1144f730c2f9a30fd7707975d781cf484eff54fec0836"
export_path = BROWSER_DIR / "export-receipt.json"
export = json.loads(export_path.read_text()) if export_path.is_file() and results_ready else {}
if export.get("status") != "COMPLETE":
    print("Local panels are optional. The numeric report works without the illustration bundle.")
else:
    if (
        export["selection_sha256"] != selection_sha
        or export["manifest_sha256"] != study["manifest_sha256"]
        or export["reconciliation_sha256"] != study["reconciliation_sha256"]
    ):
        raise ValueError("Illustrations differ from the frozen selection or reconciled study")
    bundle = json.loads(verified_bytes(BROWSER_DIR / "cases.json", export["cases_json_sha256"]))
    records = {(r["case_id"], r["sequence"]): r for r in bundle["cases"]}
    if (
        bundle["status"] != "COMPLETE"
        or len(bundle["cases"]) != 6
        or set(records) != {(c, s) for c in fixed_examples for s in ("T1", "T2")}
    ):
        raise ValueError("Expected both modalities of the three selected subjects")
    for case_id in fixed_examples:
        t1, t2 = (records[case_id, s] for s in ("T1", "T2"))
        if t1["axial_index"] != t2["axial_index"]:
            raise ValueError(f"Paired slice differs for {case_id}")
        for row in (t1, t2):
            for method in ("RAW", "SSC4"):
                value = row["whole_volume_dice"][method]
                if not np.isclose(
                    value, paired.loc[case_id, f"{method}_{row['sequence']}"], atol=1e-12, rtol=0
                ):
                    raise ValueError(f"Illustration score differs for {case_id}")
        name = f"figures/{case_id}.png"
        display(
            Image(data=verified_bytes(BROWSER_DIR / name, bundle["file_sha256"][name]), width=900)
        )

To inspect other slices, set `BROWSE_SAVED=True` below and choose a subject and sequence. This opens the retained local image, reference and masks. Leaving the switch off skips all volume reads.

In [ ]:
BROWSE_SAVED = False
CASE_ID = None  # Choose an ID listed in the local illustration metadata.
SEQUENCE = "T1"  # Change to T2 for the same held-out subject.
PLANE = "Axial"
SLICE = None  # Axial defaults to this subject’s preselected reference slice.
BROWSER_DIR = PROJECT / "data" / "brain-t1-t2-demo"
if BROWSE_SAVED:
    from ml_biomed_project.ssc import explore

    case_file = BROWSER_DIR / "cases.json"
    if not case_file.is_file():
        print("Retained illustration metadata unavailable; no images loaded.")
    else:
        records = json.loads(case_file.read_text())["cases"]
        display(pd.DataFrame(records)[["case_id", "sequence", "selection"]])
        candidates = [
            r
            for r in records
            if r["sequence"] == SEQUENCE and (CASE_ID is None or r["case_id"] == CASE_ID)
        ]
        if not candidates:
            raise ValueError("Requested retained illustration is absent")
        selected = sorted(candidates, key=lambda r: r["case_id"])[0]

        def local_path(value):
            path = (BROWSER_DIR / value).resolve()
            if not path.is_relative_to(BROWSER_DIR.resolve()):
                raise ValueError("Illustration paths must stay inside the local bundle")
            return path

        case = explore.load_case(
            local_path(selected["image"]),
            local_path(selected["reference"]),
            {name: (local_path(path), 1) for name, path in selected["predictions"].items()},
            name=f"{selected['case_id']} · {SEQUENCE}",
        )
        display(case["scores"].round(4))
        index = selected["axial_index"] if PLANE == "Axial" and SLICE is None else SLICE
        figure = explore.plot_case(case, plane=PLANE, index=index)
        figure.subplots_adjust(top=0.79)
        plt.show()
else:
    print("Saved-image browser disabled. Reporting cells read no MRI or prediction arrays.")

## 5. Reproduce preparation and fitting

Run these shell commands separately using the original local dataset. The audit and manifest builder reproduce the subject split. Review the storage pilot before full preprocessing.

Use the notebook environment from section 1. The preparation commands use POSIX signal deadlines, and the training examples use the macOS MPS device from the study. The released checkpoints also load on CPU.

```sh
python scripts/inspect_brain.py data/Task01_BrainTumour --output data/brain-illustration/audit
python scripts/freeze_brain_split.py data/Task01_BrainTumour --audit data/brain-illustration/audit --output data/brain-illustration/manifest.json
python scripts/prepare_brain.py stage --manifest data/brain-illustration/manifest.json
python scripts/prepare_brain.py plan --manifest data/brain-illustration/manifest.json
python scripts/prepare_brain.py pilot --manifest data/brain-illustration/manifest.json
python scripts/prepare_brain.py preprocess --manifest data/brain-illustration/manifest.json

# The bounded study runs these recipes sequentially under its supervisor.
python scripts/train_ssc.py 802 --workspace data/nnunet-brain --plans Brain2D --epochs 50 --seed 1729 --representation raw --device mps
python scripts/train_ssc.py 802 --workspace data/nnunet-brain --plans Brain2D --epochs 50 --seed 1729 --representation ssc --channels 4 --device mps
```

After both source fits, evaluate channel 3 once per terminal checkpoint on the same held-out subjects. All subsequent plots use saved outputs.

**Load a released model locally.** Extract either model ZIP into the project's `models/` directory. Each ZIP contains a directory named after its release tag, with `plans.json`, `dataset.json` and `fold_0/checkpoint_final.pth` inside. After the notebook setup cell has run, the following example loads the selected model from disk and verifies its checkpoint against the recorded SHA256.

```python
from ml_biomed_project.ssc.study_inference import load_study_model

release = study["released_models"]["SSC4"]  # Choose "RAW" for the intensity model.
model = load_study_model(
    PROJECT / "models" / release["model_dir"],
    checkpoint_sha256=release["checkpoint_sha256"],
    representation=release["representation"],
    epochs=release["epochs"],
    device="cpu",
)
```

## Interpretation and provenance

The sharp RAW T1-to-T2 decline seen in CrossPan also appeared here on brain tumor validation. SSC4 retained substantially more T2 overlap. Its source T1 Dice was lower, so the result supports a trade-off between source accuracy and contrast transfer in this comparison.

SSC4 encodes differences between neighboring patches. Reversing all normalized intensities preserves squared patch differences, which offers one reason this representation might transfer. Actual T1 and T2 change tissue contrast unevenly, and lesion and edema visibility can also change. No held-out reference foreground lay outside either modality’s native crop. The observed scores cannot identify a unique mechanism.

Only T1 entered fitting. The same 97 subjects and references were used on both sequences. This is development validation with one fit per recipe, a fixed epoch-50 checkpoint and no estimate of training-seed variability. Different input widths and restarted RAW augmentation streams limit how closely the training trajectories match.

Data and methods: [Medical Segmentation Decathlon](https://medicaldecathlon.com/) · [Antonelli et al., 2022](https://pmc.ncbi.nlm.nih.gov/articles/PMC9287542/) · [CrossPan notebook](ssc-domain-generalization.ipynb).